# RAG Pipeline - Exp2

* About
  * Change to simplier dataset
    * clear ground truth from the context
    * shorter context
* Learnings
  * LlamaIndex uses OpenAI by default, have to be very careful on which LLM LlamaIndex is actually using
  * chunk size = max number of tokens (text + metadata) per chunk
  * LlamaIndex has 1024 default chunk size, if metadata is long, it can create error
  * Auto Eval can't use `Llama 3.1 8B Instant 128k` cuz it's not intelligent enough to output the right format

In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
from llama_index.core import Document
import pandas as pd
import json

from utils import *

import warnings
warnings.filterwarnings('ignore')

resource module not available on Windows


### Load Data

In [2]:
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

output_dir = "fiqa_raw_text"
os.makedirs(output_dir, exist_ok=True)

rag_lst = []
documents = []  # to store documents for llamindex
for idx, record in enumerate(fiqa_eval):
    context = record['contexts'][0]
    gt = record['ground_truths'][0]
    ai0_answer = record['answer'].strip()

    with open(os.path.join(output_dir, f"{idx}.txt"), "w", encoding="utf-8") as f:
        f.write(context)
    rag_lst.append({
        'question': record['question'],
        'context': context,
        'context_ct': len(record['contexts']),
        'ground_truth': gt,
        'gt_tokens': len(gt.split()),
        'ai0_answer': ai0_answer
    })
    doc = Document(
        text=context,
        metadata={
            "question": record['question'],
            "doc_name": idx
        }
    )
    documents.append(doc)

rag_df = pd.DataFrame(rag_lst)
print(rag_df.shape, max(rag_df['context_ct']), min(rag_df['context_ct']), max(rag_df['gt_tokens']))
print(len(documents))
rag_df.head()

(30, 6) 1 1 952
30


,question,context,context_ct,ground_truth,gt_tokens,ai0_answer
0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,1,Have the check reissued to the proper payee.Ju...,131,The best way to deposit a cheque issued to an ...
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,1,Sure you can. You can fill in whatever you wa...,101,"Yes, you can send a money order from USPS as a..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,1,You're confusing a lot of things here. Company...,163,"Yes, it is possible to have one EIN doing busi..."
3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,1,"""I'm afraid the great myth of limited liabilit...",355,Applying for and receiving business credit can...
4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,1,You should probably consult an attorney. Howev...,64,If your employer has closed and you need to tr...


### Run RAG Pipeline

* Use the first config 

In [ ]:
import os
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import StorageContext, load_index_from_storage

os.environ["GROQ_API_KEY"] = os.environ["GROQ_TOKEN"]
Settings.llm = Groq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5", 
    device="cpu"
)
node_parser = setup_chunking_strategy(embed_model=Settings.embed_model)

print("Model name:", getattr(Settings.llm, "model", None))
print("Model name:", getattr(Settings.embed_model, "model_name",
                              getattr(Settings.embed_model, "model_id", None)))

Model name: llama-3.1-8b-instant
Model name: BAAI/bge-small-en-v1.5


In [8]:
if os.path.isdir("./storage"):
    storage_context = StorageContext.from_defaults(persist_dir="./storage")
    vector_index = load_index_from_storage(storage_context)
else:
    vector_index = VectorStoreIndex.from_documents(
        documents,
        node_parser=node_parser,
        show_progress=True
    )
    vector_index.storage_context.persist(persist_dir="./storage")
    print("Index saved to ./storage")

Parsing nodes:   0%|          | 0/30 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/35 [00:00<?, ?it/s]

Index saved to ./storage


In [9]:
retriever = HybridRetriever(
            vector_index,
            documents,
            top_k=1,
            alpha=0.6  # 60% weight to semantic retrieval (dense), 40% to key words retrieval (sparse)
        )
query_engine = get_query_engine(retriever, reranker=None)  # without reranker
query_engine

In [12]:
def get_rag_response(query_engine, question: str, print_query=False) -> Response:
    """
        Query the RAG system with optional query expansion
    """
    if print_query:
        print(f"\n{'='*60}")
        print(f"Query: {question}")
        print(f"{'='*60}")
        
    response = query_engine.query(question)
    retrieved_nodes = response.source_nodes
    return response, retrieved_nodes


async def _run_one(dct, query_engine):
    question = dct["question"]
    expected_answer = dct["ground_truth"]

    # run blocking call in a thread
    ai_answer, retrieved_nodes = await asyncio.to_thread(
        get_rag_response, query_engine, question
    )

    retrieved_lst = [
        {
            "metadata": n.metadata["doc_name"],
            "content": n.get_content(),
        }
        for n in retrieved_nodes
    ]

    return {
        "question": question,
        "expected_answer": expected_answer,
        "ai_answer": str(ai_answer),
        "retrieved_lst": retrieved_lst,
    }


async def run_eval_async(items, query_engine, concurrency=3):
    sem = asyncio.Semaphore(concurrency)

    async def bound_run(dct):
        async with sem:
            return await _run_one(dct, query_engine)

    tasks = [bound_run(dct) for dct in items]
    results = await asyncio.gather(*tasks)
    return results

In [16]:
# response_lst = await run_eval_async(rag_lst, query_engine, concurrency=5)
print(len(response_lst))

with open("output/fiqa_response.json", "w", encoding="utf-8") as f:
    json.dump(response_lst, f, ensure_ascii=False, indent=2)

30


### Evaluation

In [34]:
from langchain_groq import ChatGroq
import nest_asyncio
nest_asyncio.apply()

with open('eval_prompts.yaml', 'r') as file:
    prompt_versions = yaml.safe_load(file)

eval_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b", 
    temperature=0.7
)

In [35]:
eval_input = get_eval_input(response_lst)
print(eval_input.shape)
display(eval_input.head())

(30, 4)


,query,ai_answer,referenced_answer,retrieved_content
0,How to deposit a cheque issued to an associate...,You can deposit the cheque by having the assoc...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...
2,1 EIN doing business under multiple business n...,You can achieve this by creating a DBA (Doing ...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...
3,Applying for and receiving business credit,To establish a strong business credit foundati...,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...
4,401k Transfer After Business Closure,"When a business closes, it's essential to cons...",You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...


In [36]:
answer_accuracy = asyncio.run(get_answer_accuracy_output_async(eval_input, eval_llm,
                                                        prompt_versions['ac_prompt_template']))

print(answer_accuracy.shape)
display(answer_accuracy.head())

(30, 6)


,query,ai_answer,referenced_answer,retrieved_content,answer_accuracy_score,ac_reasoning
0,How to deposit a cheque issued to an associate...,You can deposit the cheque by having the assoc...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,3,"The answer explains signing the back, depositi..."
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,2,Answer is relevant and states you can use a bu...
2,1 EIN doing business under multiple business n...,You can achieve this by creating a DBA (Doing ...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,1,AI’s answer addresses the query but omits the ...
3,Applying for and receiving business credit,To establish a strong business credit foundati...,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,2,"AI answer addresses bank docs, collateral, fac..."
4,401k Transfer After Business Closure,"When a business closes, it's essential to cons...",You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,1,AI’s answer discusses 401(k) rollover options ...


In [38]:
answer_accuracy['answer_accuracy_score'].value_counts(normalize=True)

answer_accuracy_score
2    0.433333
1    0.300000
3    0.200000
0    0.066667
Name: proportion, dtype: float64